# 00.3 面向建模的 Pandas（Pandas for Modeling）

这份 notebook 的目标不是把 `Pandas` 全部学完，而是先掌握建模前最常用的数据处理动作。  

重点概念

- `DataFrame`
- 缺失值（missing values）
- 分组聚合（groupby and aggregation）
- 类别编码（categorical encoding）

## 学习目标

学完后你应该能

1. 创建并检查一个 `DataFrame`
2. 选择列、筛选行、理解 `loc`
3. 处理缺失值
4. 用 `groupby` 做简单统计
5. 用 `get_dummies` 做基础类别编码
6. 把清洗后的数据交给后续模型

In [ ]:
import numpy as np
import pandas as pd

## 1. 创建与检查 `DataFrame`

`DataFrame` 可以理解成“带列名的二维表格”。  


In [ ]:
df = pd.DataFrame(
    {
        "student_id": [101, 102, 103, 104, 105, 106],
        "hours": [1.5, 3.0, 2.2, 1.0, 4.1, 2.8],
        "attendance": [0.70, 0.90, 0.80, 0.60, 0.95, 0.85],
        "sleep_hours": [6.0, 7.0, np.nan, 5.5, 7.5, 6.8],
        "city": ["A", "B", "A", "B", "C", None],
        "passed": [0, 1, 1, 0, 1, 1],
    }
)

print("前几行 / head:")
print(df.head())
print()
print("形状 / shape:", df.shape)
print("列名 / columns:", list(df.columns))

In [ ]:
print("数据概览 / info:")
print(df.info())
print()
print("描述统计 / describe:")
print(df.describe(include="all"))

重点理解

- `df.info()` 看类型和缺失值
- `df.describe()` 看统计概况（shows summary statistics）
- `shape` 看样本数和列数

## 2. 选列与筛行

机器学习前的数据处理，最常见的动作就是选列和筛行。  


In [ ]:
feature_cols = ["hours", "attendance", "sleep_hours"]
features = df[feature_cols]
target = df["passed"]

high_attendance = df[df["attendance"] >= 0.85]
selected_rows = df.loc[df["city"].isin(["A", "B"]), ["student_id", "city", "passed"]]

print("特征列 / feature columns:")
print(features)
print()
print("目标列 / target:")
print(target)
print()
print("高出勤样本 / high attendance rows:")
print(high_attendance)
print()
print("按条件选择的行列 / selected rows and columns:")
print(selected_rows)

In [ ]:
# 练习 1
# 目标
# 1. 选出 city 为 A 的所有样本
# 2. 只保留 hours 和 passed 两列
# 3. 再筛出 hours > 2 的样本

# city_a =
# result =

# print(result)

In [ ]:
# 练习 1 参考答案

city_a = df[df["city"] == "A"]
result = city_a.loc[city_a["hours"] > 2, ["hours", "passed"]]
print(result)

## 3. 缺失值

真实数据里，缺失值非常常见。  

处理缺失值的第一步不是“立刻填”，而是先回答两个问题：  

1. 哪些列缺失？/ Which columns are missing values?
2. 缺失值该删除、填补还是单独编码？/ Should they be dropped, imputed, or encoded separately?

In [ ]:
print("每列缺失值数量 / missing values per column:")
print(df.isna().sum())
print()

df_filled = df.copy()
df_filled["sleep_hours"] = df_filled["sleep_hours"].fillna(df_filled["sleep_hours"].mean())
df_filled["city"] = df_filled["city"].fillna("Unknown")

print("填补后的数据 / after filling:")
print(df_filled)
print()
print("填补后缺失值 / missing values after filling:")
print(df_filled.isna().sum())

常见策略

- numeric columns: 均值、中位数、固定值
- categorical columns: 众数、`Unknown`、单独一类（mode, `Unknown`, or a dedicated category）
- too much missingness: 可能要删列

In [ ]:
# 练习 2
# 请实现 clean_student_df(dataframe)
# Implement clean_student_df(dataframe)
#
# 要求
# 1. 复制原表
# 2. 用均值填补 sleep_hours
# 3. 用 "Unknown" 填补 city
# 4. 返回清洗后的 DataFrame

def clean_student_df(dataframe):
    # TODO
    pass


# print(clean_student_df(df))

In [ ]:
# 练习 2 参考答案

def clean_student_df_solution(dataframe):
    result = dataframe.copy()
    result["sleep_hours"] = result["sleep_hours"].fillna(result["sleep_hours"].mean())
    result["city"] = result["city"].fillna("Unknown")
    return result


print(clean_student_df_solution(df))

## 4. 分组聚合

`groupby` 的作用是：按某一列或某几列分组，再做统计。  

它在数据理解阶段非常有用。  


In [ ]:
city_summary = (
    df_filled.groupby("city")
    .agg(
        avg_hours=("hours", "mean"),
        avg_attendance=("attendance", "mean"),
        pass_rate=("passed", "mean"),
        num_students=("student_id", "count"),
    )
    .sort_values("pass_rate", ascending=False)
)

print("按城市汇总 / summary by city:")
print(city_summary)

这里的 `pass_rate` 直接用均值，是因为标签是 0/1。  

这是一种很常见的小技巧。  


## 5. 类别编码

模型通常不能直接使用字符串类别，所以需要编码。  

最基础的方法是 one-hot encoding，一般可用 `pd.get_dummies`。  


In [ ]:
encoded = pd.get_dummies(df_filled, columns=["city"], dtype=int)

print("编码后 / after encoding:")
print(encoded)
print()
print("编码后列名 / encoded columns:")
print(list(encoded.columns))

## 6. 面向建模的最小流程

很多时候你真正需要的是：  

1. 清洗数据
2. 做必要编码
3. 切出特征和标签
4. 转成后续模型能接收的格式

In [ ]:
model_df = pd.get_dummies(clean_student_df_solution(df), columns=["city"], dtype=float)
X = model_df.drop(columns=["student_id", "passed"])
y = model_df["passed"]

print("X / features:")
print(X)
print()
print("y / target:")
print(y)
print()
print("X 的形状 / X shape:", X.shape)
print("y 的形状 / y shape:", y.shape)

In [ ]:
# 练习 3
# 实现 prepare_features_and_target(dataframe)
# Implement prepare_features_and_target(dataframe)
#
# 要求
# 1. 先清洗缺失值
# 2. 对 city 做 one-hot 编码
# 3. 删除 student_id 和 passed
# 4. 返回 X, y

def prepare_features_and_target(dataframe):
    # TODO
    pass


# X_out, y_out = prepare_features_and_target(df)
# print(X_out)
# print(y_out)

In [ ]:
# 练习 3 参考答案

def prepare_features_and_target_solution(dataframe):
    cleaned = clean_student_df_solution(dataframe)
    encoded = pd.get_dummies(cleaned, columns=["city"], dtype=float)
    X = encoded.drop(columns=["student_id", "passed"])
    y = encoded["passed"]
    return X, y


X_out, y_out = prepare_features_and_target_solution(df)
print(X_out)
print(y_out)

## 7. 小结

本节的核心不是背 `Pandas` API，而是掌握建模前的最小工作流。  

你现在应该能回答

1. `DataFrame` 和 `Series` 的关系是什么？
2. 为什么先检查缺失值再填补？
3. 为什么模型前常要做类别编码？
4. 为什么 `groupby` 对数据理解有帮助？

下一步建议

- 进入可视化 notebook，把数据和训练结果“画出来”